# Prediction du Comportement des Clients avec l'IA dans une Banque
## Scoring Bancaire — ANI-IA 4

**Objectifs :**
- Comparer differents types de classificateurs
- Creer un ensemble de modeles (Voting Classifier)
- Creer un classificateur personnalise base sur un reseau de neurones (MLP)
- Classer les clients en fonction des modeles developpes

**Dataset :** [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

---


## 1. Installation des dependances et Imports

In [10]:
# Installation (decommentier si necessaire)
!pip install ucimlrepo scikit-learn pandas numpy matplotlib seaborn imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, BaggingClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Style des graphiques
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style("whitegrid")

print("Tous les imports realises avec succes !")


  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.9-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -----------------------------------

ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'C:\\Users\\LENOVO\\AppData\\Local\\Temp\\pip-unpack-eykhygz4\\pandas-3.0.3-cp311-cp311-win_amd64.whl'
Check the permissions.


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'pandas'

---
## 2. Chargement et Exploration des Donnees

Le dataset **Bank Marketing** contient des informations sur les clients d'une banque portugaise et les resultats de campagnes de marketing telephonique direct.


In [4]:
from pathlib import Path
from zipfile import ZipFile
from urllib.request import urlretrieve

project_dir = Path.cwd()
local_file = project_dir / 'bank_marketing_uci.csv'
archive_file = project_dir / 'bank.zip'
source_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip'

if local_file.exists():
    df = pd.read_csv(local_file)
    print(f"Dataset charge depuis le fichier local : {local_file}")
else:
    try:
        if not archive_file.exists():
            print(f"Telechargement du dataset depuis UCI : {source_url}")
            urlretrieve(source_url, archive_file)

        with ZipFile(archive_file) as archive:
            csv_name = 'bank-full.csv'
            with archive.open(csv_name) as csv_file:
                df = pd.read_csv(csv_file, sep=';')

        df.to_csv(local_file, index=False)
        print("Dataset telecharge depuis UCI et sauvegarde localement")
        print(f"Fichier cree : {local_file}")
    except Exception as error:
        raise RuntimeError(
            "Impossible de telecharger le dataset reel depuis UCI. Verifiez la connexion Internet."
        ) from error

print(f"\nShape du dataset : {df.shape}")
print(f"Colonnes : {list(df.columns)}")


NameError: name 'pd' is not defined

In [ ]:
# Apercu des premieres lignes
df.head(5)

In [ ]:
print("=" * 55)
print("   INFORMATIONS GENERALES DU DATASET")
print("=" * 55)
print(df.info())


In [ ]:
df.shape

### Renommage des colonnes en francais

In [ ]:
# Renommer les colonnes en francais
colonnes_francais = {
    'age': 'age',
    'job': 'emploi',
    'marital': 'etat_civil',
    'education': 'education',
    'default': 'credit_defaut',
    'balance': 'solde',
    'housing': 'pret_immobilier',
    'loan': 'pret_personnel',
    'contact': 'type_contact',
    'day': 'jour',
    'month': 'mois',
    'duration': 'duree',
    'campaign': 'campagne',
    'pdays': 'jours_depuis_contact',
    'previous': 'contacts_precedents',
    'poutcome': 'resultat_precedent',
    'y': 'souscription'
}

df = df.rename(columns=colonnes_francais)

num_cols = ['age', 'solde', 'duree', 'campagne', 'jours_depuis_contact', 'contacts_precedents']
cat_cols = ['emploi', 'education', 'etat_civil', 'resultat_precedent', 'pret_immobilier', 'pret_personnel']

print("Colonnes renommees :")
print(list(df.columns))


In [ ]:
df.isna().sum()

In [ ]:
# Statistiques descriptives des variables numeriques
df.describe().round(2)

---
## 3. Analyse Exploratoire des Donnees (EDA)

### 3.1 Analyse univariee


In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['souscription'].value_counts()
colors = ['#2E75B6','#C55A11']
axes[0].pie(counts, labels=['Non souscrit (no)','Souscrit (yes)'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Distribution de la variable cible', fontweight='bold')

axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_xlabel('Souscription (y)')
axes[1].set_ylabel("Nombre d'observations")
axes[1].set_title('Desequilibre des classes', fontweight='bold')
for i, (k, v) in enumerate(counts.items()):
    axes[1].text(i, v+200, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nTaux de souscription : {(df['souscription']=='yes').mean()*100:.1f}%")


### 3.2 Analyse bivariee des variables numeriques par classe

In [ ]:
# Distribution des variables numeriques par classe
# Note : 'duree' est incluse ici pour l'EDA uniquement.
# Elle sera supprimee avant la modelisation (data leakage).
num_cols = ['age', 'solde', 'duree', 'campagne', 'jours_depuis_contact', 'contacts_precedents', 'jour']
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[df['souscription']=='no'][col].hist(ax=axes[i], alpha=0.6, color='#2E75B6',
                                 label='No', bins=30, density=True)
    df[df['souscription']=='yes'][col].hist(ax=axes[i], alpha=0.6, color='#C55A11',
                                  label='Yes', bins=30, density=True)
    axes[i].set_title(f'Distribution de {col}', fontweight='bold')
    axes[i].legend()

for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')

plt.suptitle('Distribution des variables numeriques par classe', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('num_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


### Test de Welch (t-test) — Variables numeriques vs Cible

In [ ]:
from scipy import stats

num_cols_test = ['age', 'solde', 'campagne', 'jours_depuis_contact', 'contacts_precedents', 'jour']

group_no  = df[df['souscription'] == 'no']
group_yes = df[df['souscription'] == 'yes']

print(f"Nombre de clients — No : {len(group_no):,}  |  Yes : {len(group_yes):,}")
print("\n" + "="*45)
print("   Moyennes par classe")
print("="*45)
print(f"{'Variable':<25} {'Moy. NO':>8} {'Moy. YES':>9}")
print("-"*45)

resultats = {}
for col in num_cols_test:
    no_vals  = group_no[col].dropna()
    yes_vals = group_yes[col].dropna()
    t_stat, p_val = stats.ttest_ind(no_vals, yes_vals, equal_var=False)
    resultats[col] = {
        'moy_no': no_vals.mean(), 'moy_yes': yes_vals.mean(),
        'std_no': no_vals.std(),  'std_yes': yes_vals.std(),
        't_stat': t_stat, 'p_val': p_val, 'sig': p_val < 0.05,
    }
    print(f"{col:<25} {no_vals.mean():>8.2f} {yes_vals.mean():>9.2f}")
print("="*45)

COLORS = {'no': '#2E75B6', 'yes': '#C55A11'}
BINS   = 35
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols_test):
    ax = axes[i]
    r  = resultats[col]
    no_vals  = group_no[col].dropna()
    yes_vals = group_yes[col].dropna()
    ax.hist(no_vals,  bins=BINS, density=True, alpha=0.60, color=COLORS['no'],  label='No',  edgecolor='white', linewidth=0.4)
    ax.hist(yes_vals, bins=BINS, density=True, alpha=0.60, color=COLORS['yes'], label='Yes', edgecolor='white', linewidth=0.4)
    ax.axvline(r['moy_no'],  color=COLORS['no'],  linestyle='--', linewidth=1.8, label=f"Moy. No  = {r['moy_no']:.1f}")
    ax.axvline(r['moy_yes'], color=COLORS['yes'], linestyle='--', linewidth=1.8, label=f"Moy. Yes = {r['moy_yes']:.1f}")
    sig_str = "Significatif" if r['sig'] else "Non significatif"
    color_title = '#375623' if r['sig'] else '#C00000'
    ax.set_title(f"{col}  —  {sig_str}", fontsize=10, fontweight='bold', color=color_title)
    ax.text(0.98, 0.97, f"t = {r['t_stat']:.2f}\np = {r['p_val']:.2e}",
            transform=ax.transAxes, fontsize=8.5, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='#E8F4FD' if r['sig'] else '#FFF0F0',
                      edgecolor='#2E75B6' if r['sig'] else '#C55A11', linewidth=0.8))
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("Densite", fontsize=9)
    ax.legend(fontsize=7.5, loc='upper right')
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle("t-test de Welch — Variables numeriques vs Cible\nBank Marketing Dataset",
             fontsize=13, fontweight='bold', y=1.01, color='#1F4E79')
plt.tight_layout()
plt.savefig('ttest_histogrammes.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Taux de souscription par variable categorielle

In [ ]:
cat_cols_plot = ['emploi', 'education', 'etat_civil', 'resultat_precedent', 'pret_immobilier', 'pret_personnel']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols_plot):
    rate = df.groupby(col)['souscription'].apply(lambda x: (x=='yes').mean() * 100).sort_values(ascending=False)
    axes[i].bar(range(len(rate)), rate.values, color='#2E75B6', edgecolor='white')
    axes[i].set_xticks(range(len(rate)))
    axes[i].set_xticklabels(rate.index, rotation=35, ha='right', fontsize=8)
    axes[i].set_ylabel('Taux de souscription (%)')
    axes[i].set_title(f'Taux par {col}', fontweight='bold')
    axes[i].axhline(y=(df['souscription']=='yes').mean()*100, color='#C55A11',
                    linestyle='--', linewidth=1.5, label='Moyenne globale')
    axes[i].legend(fontsize=8)

plt.suptitle('Taux de souscription par variable categorielle', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cat_rates.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap de contingence Yes/No par modalite
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols_plot):
    ax = axes[i]
    table = pd.crosstab(df[col], df['souscription'], normalize='index') * 100
    sns.heatmap(table, annot=True, fmt='.1f', cmap='Blues',
                linewidths=0.5, linecolor='white', cbar=False, ax=ax, annot_kws={'size': 9})
    ax.set_title(col, fontsize=11, fontweight='bold', color='#1F4E79')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=9)

fig.suptitle('Heatmap — Proportion Yes/No par modalite (%)', fontsize=13, fontweight='bold', color='#1F4E79')
plt.tight_layout()
plt.savefig('heatmap_categories.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.4 Matrice de correlation des variables numeriques

In [ ]:
df_num = df[['age', 'solde', 'campagne', 'jours_depuis_contact', 'contacts_precedents', 'jour']].copy()
corr = df_num.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de Correlation — Variables numeriques (sans duree)', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4. Preparation des Donnees

### Etapes :
1. Suppression de `duree` (data leakage — variable inconnue avant l'appel)
2. Encodage des variables categorielles avec sauvegarde des encodeurs
3. Encodage cyclique (sin/cos) de la colonne `mois`
4. Division Train / Test (80% / 20%, stratifiee)
5. Normalisation StandardScaler
6. Gestion du desequilibre avec SMOTE


### 4.1 Suppression de `duree` — data leakage

La variable `duree` (duree de l'appel en secondes) n'est connue qu'**apres** l'appel telephonique.
Un modele de scoring reel ne peut pas l'utiliser comme feature d'entree.
Le dataset UCI lui-meme signale ce probleme : la garder produirait un modele inutilisable en production.


In [ ]:
df_enc = df.copy()

# Suppression de 'duree' : data leakage
df_enc = df_enc.drop(columns=['duree'])

print("'duree' supprimee — data leakage evite")
print(f"Features restantes : {df_enc.shape[1] - 1}")
print(f"Colonnes : {list(df_enc.columns)}")


### 4.2 Encodage des variables categorielles

**LabelEncoder** pour les colonnes categorielles avec sauvegarde des encodeurs.
**Encodage ordinal** pour `type_contact` (hierarchie de qualite de contact).
**Encodage cyclique sin/cos** pour `mois` : les mois forment un cycle (decembre est proche de janvier),
ce que l'encodage ordinal simple ne capture pas. On utilise deux colonnes (`mois_sin`, `mois_cos`)
pour identifier chaque mois de facon unique sur le cercle.


In [ ]:
# Encodage des variables categorielles avec sauvegarde des encodeurs
cat_cols = ['emploi', 'education', 'etat_civil',
            'resultat_precedent', 'pret_immobilier', 'pret_personnel', 'credit_defaut']

encoders = {}   # Sauvegarde pour reutilisation en production
for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    encoders[col] = le

# Encodage de la variable cible
df_enc['souscription'] = (df_enc['souscription'] == 'yes').astype(int)

# Encodage ordinal de type_contact (hierarchie de qualite)
mapping_contact = {
    'unknown':   1,
    'telephone': 2,
    'cellular':  3
}
df_enc['type_contact'] = df_enc['type_contact'].map(mapping_contact)

# Encodage du mois : ordinal + cyclique (sin/cos) pour capturer la circularite
mapping_mois = {
    'jan': 1,  'feb': 2,  'mar': 3,  'apr': 4,
    'may': 5,  'jun': 6,  'jul': 7,  'aug': 8,
    'sep': 9,  'oct': 10, 'nov': 11, 'dec': 12
}
df_enc['mois'] = df_enc['mois'].str.lower().str.strip().map(mapping_mois)
df_enc['mois_sin'] = np.sin(2 * np.pi * df_enc['mois'] / 12)
df_enc['mois_cos'] = np.cos(2 * np.pi * df_enc['mois'] / 12)

print("Encodage termine")
print(f"Distribution de la cible encodee :")
print(df_enc['souscription'].value_counts())
print(f"\nTaux positif : {df_enc['souscription'].mean()*100:.1f}%")
print(f"Valeurs manquantes apres encodage : {df_enc.isna().sum().sum()}")


In [ ]:
df_enc.head(5)

### 4.3 Division Train / Test

In [ ]:
# Division Train / Test
X = df_enc.drop('souscription', axis=1)
y = df_enc['souscription']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserve la proportion des classes
)

# Normalisation (fit UNIQUEMENT sur train)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Division effectuee")
print(f"   Train : {X_train.shape[0]:,} observations ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Test  : {X_test.shape[0]:,} observations ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"   Features : {X_train.shape[1]}")
print(f"\n   Taux positif train : {y_train.mean()*100:.1f}%")
print(f"   Taux positif test  : {y_test.mean()*100:.1f}%")


### 4.4 Gestion du desequilibre — SMOTE

Le dataset contient ~88% de "no" et ~12% de "yes". Ce fort desequilibre pousse les modeles
a ignorer la classe minoritaire. **SMOTE** (Synthetic Minority Over-sampling Technique)
genere des exemples synthetiques de la classe "yes" uniquement sur le jeu d'entrainement,
ce qui preserve l'integrite du jeu de test.


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])
    from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print("SMOTE applique")
print(f"   Avant SMOTE — No: {(y_train==0).sum():,}  |  Yes: {(y_train==1).sum():,}")
print(f"   Apres SMOTE — No: {(y_train_sm==0).sum():,}  |  Yes: {(y_train_sm==1).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (y_plot, title) in zip(axes, [
    (y_train,    "Avant SMOTE (train)"),
    (y_train_sm, "Apres SMOTE (train)")
]):
    counts = pd.Series(y_plot).value_counts()
    ax.bar(['No (0)', 'Yes (1)'], counts.values,
           color=['#2E75B6', '#C55A11'], edgecolor='white', linewidth=1.5)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel("Nombre d'observations")

plt.suptitle("Effet du SMOTE sur la distribution des classes", fontweight='bold')
plt.tight_layout()
plt.savefig('smote_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 5. Implementation et Comparaison des Classificateurs

### Modeles implementes :
| # | Modele | Type | Normalisation requise |
|---|--------|------|-----------------------|
| 1 | Logistic Regression | Lineaire | necessite |
| 2 | Decision Tree | Arbre | ne necessite pas |
| 3 | Random Forest | Ensemble (Bagging) | ne necessite pas |
| 4 | Gradient Boosting | Ensemble (Boosting) | ne necessite pas |
| 5 | AdaBoost | Ensemble (Boosting) | ne necessite pas |
| 6 | Naive Bayes | Probabiliste | necessite |
| 7 | K-Nearest Neighbors | Instance-based | necessite |
| 8 | MLP (64-32) | Reseau de neurones | necessite |
| 9 | MLP (128-64-32) | Reseau de neurones | necessite |
| 10 | Voting Ensemble | Ensemble (Voting) | — |

Les modeles necessitant une normalisation utilisent les donnees apres SMOTE (`X_train_sm`, `y_train_sm`).


In [ ]:
# Definition des modeles
classifiers = {
    'Logistic Regression': (
        LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
        True   # utilise donnees normalisees + SMOTE
    ),
    'Decision Tree': (
        DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
        False
    ),
    'Random Forest': (
        RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               n_jobs=-1, random_state=42),
        False
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=3, random_state=42),
        False
    ),
    'AdaBoost': (
        AdaBoostClassifier(n_estimators=100, random_state=42),
        False
    ),
    'Naive Bayes': (
        GaussianNB(),
        True
    ),
    'KNN (k=11)': (
        KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
        True
    ),
    'MLP (64-32)': (
        MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                      solver='adam', max_iter=300, early_stopping=True,
                      validation_fraction=0.1, random_state=42),
        True
    ),
    'MLP (128-64-32)': (
        MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                      solver='adam', alpha=0.001, learning_rate='adaptive',
                      max_iter=500, early_stopping=True, validation_fraction=0.1,
                      n_iter_no_change=20, random_state=42),
        True
    ),
}

print(f"{len(classifiers)} modeles definis et prets a l'entrainement")


In [ ]:
# Entrainement et evaluation de tous les modeles
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("=" * 70)
print(f"{'Modele':<22} | {'CV AUC':>8} | {'Test AUC':>9} | {'Accuracy':>9} | {'F1':>7}")
print("=" * 70)

for name, (clf, use_sc) in classifiers.items():
    # Modeles normalises utilisent SMOTE, les arbres utilisent le train original
    Xtr = X_train_sm   if use_sc else X_train
    Ytr = y_train_sm   if use_sc else y_train
    Xte = X_test_sc    if use_sc else X_test

    cv_auc = cross_val_score(clf, Xtr, Ytr, cv=cv, scoring='roc_auc')

    clf.fit(Xtr, Ytr)
    y_pred = clf.predict(Xte)
    y_prob = clf.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)

    results[name] = {
        'clf': clf, 'use_sc': use_sc,
        'cv_auc': cv_auc.mean(), 'cv_std': cv_auc.std(),
        'test_acc': acc, 'test_auc': auc, 'test_f1': f1,
        'y_prob': y_prob, 'y_pred': y_pred,
    }

    flag = " <-- meilleur" if auc == max([v.get('test_auc', 0) for v in results.values()]) else ""
    print(f"{name:<22} | {cv_auc.mean():.4f}+/-{cv_auc.std():.3f} | {auc:.4f}    | {acc:.4f}   | {f1:.4f}{flag}")

print("=" * 70)
print("Entrainement termine !")


---
## 6. Ensemble de Modeles — Voting Classifier

Le **Voting Classifier (soft)** combine plusieurs modeles en calculant la moyenne de leurs probabilites predites.
Il exploite la complementarite des modeles pour reduire biais et variance.


In [ ]:
# Creation du Voting Ensemble (4 modeles complementaires)
voting = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                       n_jobs=-1, random_state=42)),
        ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('ada', AdaBoostClassifier(n_estimators=100, random_state=42)),
        ('lr',  LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)),
    ],
    voting='soft'      # moyenne des probabilites (meilleur que vote dur)
)

print("Entrainement du Voting Ensemble...")
voting.fit(X_train, y_train)

yv_pred = voting.predict(X_test)
yv_prob = voting.predict_proba(X_test)[:, 1]

acc_v = accuracy_score(y_test, yv_pred)
auc_v = roc_auc_score(y_test, yv_prob)
f1_v  = f1_score(y_test, yv_pred)

results['Voting Ensemble'] = {
    'clf': voting, 'use_sc': False,
    'cv_auc': 0, 'cv_std': 0,
    'test_acc': acc_v, 'test_auc': auc_v, 'test_f1': f1_v,
    'y_prob': yv_prob, 'y_pred': yv_pred,
}

print(f"\nVoting Ensemble entraine !")
print(f"   AUC-ROC  : {auc_v:.4f}")
print(f"   Accuracy : {acc_v:.4f}")
print(f"   F1-Score : {f1_v:.4f}")


---
## 7. Classificateur Personnalise — Reseau de Neurones (MLP)

### Architecture : `Entree → Dense(128, ReLU) → Dense(64, ReLU) → Dense(32, ReLU) → Sortie(Sigmoide)`

| Couche | Taille | Activation | Role |
|--------|--------|-----------|------|
| Entree | nb features | — | Variables normalisees |
| Couche 1 | 128 | ReLU | Representations haut niveau |
| Couche 2 | 64 | ReLU | Compression |
| Couche 3 | 32 | ReLU | Representation finale |
| Sortie | 1 | Sigmoide | P(souscription) in [0,1] |


In [ ]:
# Reseau de neurones personnalise complet
mlp_custom = MLPClassifier(
    # Architecture profonde
    hidden_layer_sizes=(128, 64, 32),

    # Fonction d'activation : ReLU
    # f(x) = max(0, x) — evite le probleme du gradient vanishing
    activation='relu',

    # Optimiseur : Adam (Adaptive Moment Estimation)
    # Adapte le learning rate pour chaque parametre independamment
    solver='adam',

    # Regularisation L2 : penalise les grands poids → reduit surapprentissage
    alpha=0.001,

    # Learning rate adaptatif : reduit automatiquement si la loss stagne
    learning_rate='adaptive',

    batch_size='auto',
    max_iter=500,

    # Arret precoce : surveille la perte sur un val set interne
    early_stopping=True,
    validation_fraction=0.1,   # 10% du train pour la validation
    n_iter_no_change=20,       # patience = 20 epochs sans amelioration

    random_state=42,
    verbose=False
)

print("Entrainement du MLP personnalise (128-64-32)...")
mlp_custom.fit(X_train_sm, y_train_sm)

y_pred_mlp = mlp_custom.predict(X_test_sc)
y_prob_mlp = mlp_custom.predict_proba(X_test_sc)[:, 1]

print(f"\nMLP entraine en {mlp_custom.n_iter_} iterations")
print(f"\n{'='*50}")
print("   RAPPORT DE CLASSIFICATION — MLP (128-64-32)")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_mlp,
      target_names=['Non Souscrit (0)', 'Souscrit (1)']))


In [ ]:
# Courbe d'apprentissage (loss) du MLP
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(mlp_custom.loss_curve_, color='#2E75B6', linewidth=2, label='Loss entrainement')
if hasattr(mlp_custom, 'validation_scores_') and mlp_custom.validation_scores_:
    ax2 = ax.twinx()
    ax2.plot(mlp_custom.validation_scores_, color='#C55A11', linewidth=2,
             linestyle='--', label='Score validation')
    ax2.set_ylabel('Score validation', color='#C55A11')
    ax2.legend(loc='lower right')

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log-loss)', color='#2E75B6')
ax.set_title("Courbe d'apprentissage du MLP (128-64-32)", fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mlp_learning_curve.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 8. Visualisations et Comparaison Globale


In [ ]:
# Tableau recapitulatif des performances
summary = pd.DataFrame({
    name: {
        'CV AUC':   round(v['cv_auc'], 4),
        'Test AUC': round(v['test_auc'], 4),
        'Accuracy': round(v['test_acc'], 4),
        'F1-Score': round(v['test_f1'], 4),
    }
    for name, v in results.items()
}).T.sort_values('Test AUC', ascending=False)

print("=" * 65)
print("        COMPARAISON DES MODELES — RESULTATS SUR TEST SET")
print("=" * 65)
print(summary.to_string())
print("=" * 65)
print(f"\nMeilleur modele (AUC) : {summary['Test AUC'].idxmax()}")
print(f"   AUC-ROC  = {summary['Test AUC'].max():.4f}")
print(f"   F1-Score = {summary.loc[summary['Test AUC'].idxmax(), 'F1-Score']:.4f}")
summary


In [ ]:
# Graphique comparatif : AUC, Accuracy, F1
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['Test AUC', 'Accuracy', 'F1-Score']
titles  = ['AUC-ROC', 'Accuracy', 'F1-Score']
colors  = ['#2E75B6', '#375623', '#C55A11']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    data = summary[metric].sort_values(ascending=True)
    bars = ax.barh(data.index, data.values, color=color, edgecolor='white',
                   linewidth=1.2, height=0.6)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlim(data.min() * 0.97, data.max() * 1.02)
    for bar, val in zip(bars, data.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    best_idx = list(data.index).index(data.idxmax())
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Comparaison des Classificateurs — Bank Marketing Dataset',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Courbes ROC de tous les modeles
fig, ax = plt.subplots(figsize=(10, 7))

cmap = plt.cm.get_cmap('tab10')
for i, (name, v) in enumerate(sorted(results.items(),
                                      key=lambda x: x[1]['test_auc'], reverse=True)):
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    ax.plot(fpr, tpr, linewidth=2, color=cmap(i),
            label=f"{name}  (AUC = {v['test_auc']:.4f})")

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Aleatoire (AUC = 0.50)')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
ax.set_title('Courbes ROC — Comparaison de tous les classificateurs', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Matrices de confusion pour les 3 meilleurs modeles
top3 = summary['Test AUC'].head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, top3):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'],
                linewidths=0.5, annot_kws={'size': 12})
    ax.set_xlabel('Predit', fontweight='bold')
    ax.set_ylabel('Reel', fontweight='bold')
    ax.set_title(f'{name}\nAUC = {results[name]["test_auc"]:.4f}', fontweight='bold')

plt.suptitle('Matrices de Confusion — Top 3 Modeles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 8bis. Optimisation des Hyperparametres — GridSearchCV

Le **GridSearchCV** explore systematiquement toutes les combinaisons de parametres
possibles et selectionne celle qui maximise le score AUC-ROC en validation croisee.
Cela garantit que le meilleur modele utilise des hyperparametres calibres sur les donnees,
et non choisis arbitrairement.


In [ ]:
print("GridSearchCV en cours (environ 1-2 min)...")

param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [5, 10, None],
    'min_samples_split': [2, 5],
    'class_weight':      ['balanced']
}

gs = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid = param_grid,
    cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)

gs.fit(X_train, y_train)

best_rf   = gs.best_estimator_
y_pred_gs = best_rf.predict(X_test)
y_prob_gs = best_rf.predict_proba(X_test)[:, 1]

print(f"\nGridSearchCV termine !")
print(f"   Meilleurs parametres : {gs.best_params_}")
print(f"   Meilleur CV AUC      : {gs.best_score_:.4f}")
print(f"\n{'='*50}")
print(f"   AUC-ROC  sur test : {roc_auc_score(y_test, y_prob_gs):.4f}")
print(f"   Accuracy sur test : {accuracy_score(y_test, y_pred_gs):.4f}")
print(f"   F1-Score sur test : {f1_score(y_test, y_pred_gs):.4f}")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_gs,
      target_names=['Non Souscrit (0)', 'Souscrit (1)']))


In [ ]:
# Visualisation des resultats du GridSearch
results_gs = pd.DataFrame(gs.cv_results_)
top_results = results_gs.sort_values('mean_test_score', ascending=False).head(8)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    range(len(top_results)),
    top_results['mean_test_score'],
    xerr=top_results['std_test_score'],
    color='#2E75B6', edgecolor='white', linewidth=1.2
)
labels = [
    f"n={r['param_n_estimators']} | depth={r['param_max_depth']} | split={r['param_min_samples_split']}"
    for _, r in top_results.iterrows()
]
ax.set_yticks(range(len(top_results)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('CV AUC-ROC moyen')
ax.set_title('Top 8 combinaisons — GridSearchCV (Random Forest)', fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('gridsearch_results.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 9. Importance des Variables (Feature Importance)

Calculee via le **Random Forest optimise** : reduction moyenne de l'impurete de Gini apportee par chaque variable.
Note : `duree` a ete supprimee (data leakage), les resultats refletent donc les vraies variables utilisables en production.


In [ ]:
# Extraction de l'importance via le Random Forest optimise (GridSearchCV)
feat_importance = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top variables les plus importantes :")
print("-" * 40)
for feat, imp in feat_importance.head(10).items():
    bar = "#" * int(imp * 100)
    print(f"  {feat:<25} {imp:.4f}  {bar}")


In [ ]:
# Visualisation de l'importance des variables
fig, ax = plt.subplots(figsize=(10, 6))

colors_fi = ['#C55A11' if i < 3 else '#2E75B6' for i in range(len(feat_importance))]
bars = ax.barh(feat_importance.index[::-1], feat_importance.values[::-1],
               color=colors_fi[::-1], edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, feat_importance.values[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Importance (reduction Gini moyenne)', fontsize=11)
ax.set_title("Importance des Variables — Random Forest optimise (top variables en orange)",
             fontweight='bold', fontsize=13)

top_patch   = mpatches.Patch(color='#C55A11', label='Top 3 variables')
other_patch = mpatches.Patch(color='#2E75B6', label='Autres variables')
ax.legend(handles=[top_patch, other_patch], loc='lower right')
ax.grid(True, axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 10. Prediction sur un Nouveau Client

Demonstration de l'utilisation du meilleur modele pour scorer un nouveau client.
Le client fictif n'inclut pas `duree` puisque cette variable n'est pas disponible avant l'appel.


In [ ]:
# Definition d'un client fictif (memes colonnes que l'entrainement, sans 'duree')
nouveau_client_brut = pd.DataFrame([{
    'age':                  42,
    'emploi':               'management',
    'etat_civil':           'married',
    'education':            'tertiary',
    'credit_defaut':        'no',
    'solde':                2500,
    'pret_immobilier':      'no',
    'pret_personnel':       'no',
    'type_contact':         'cellular',
    'jour':                 15,
    'mois':                 'jun',
    'campagne':             1,
    'jours_depuis_contact': -1,
    'contacts_precedents':  0,
    'resultat_precedent':   'unknown',
}])

nouveau_client = nouveau_client_brut.copy()

# Appliquer le meme encodage que pour l'entrainement
for col in ['emploi', 'education', 'etat_civil', 'resultat_precedent',
            'pret_immobilier', 'pret_personnel', 'credit_defaut']:
    nouveau_client[col] = encoders[col].transform(nouveau_client[col])

nouveau_client['type_contact'] = nouveau_client['type_contact'].map(mapping_contact)
nouveau_client['mois']         = nouveau_client['mois'].map(mapping_mois)
nouveau_client['mois_sin']     = np.sin(2 * np.pi * nouveau_client['mois'] / 12)
nouveau_client['mois_cos']     = np.cos(2 * np.pi * nouveau_client['mois'] / 12)

# S'assurer que les colonnes sont dans le meme ordre que X
nouveau_client = nouveau_client[X.columns]

print("=" * 55)
print("   SCORING DU NOUVEAU CLIENT")
print("=" * 55)
print(f"  Age      : 42 ans")
print(f"  Emploi   : Management")
print(f"  Balance  : 2500 euros")
print(f"  Campagne : 1er contact")
print(f"  Mois     : Juin")
print("=" * 55)

best_model_name = summary['Test AUC'].idxmax()
best_clf_pred   = results[best_model_name]['clf']
use_sc_pred     = results[best_model_name]['use_sc']

X_new = scaler.transform(nouveau_client) if use_sc_pred else nouveau_client
proba = best_clf_pred.predict_proba(X_new)[0, 1]
pred  = best_clf_pred.predict(X_new)[0]

print(f"\nMeilleur modele : {best_model_name}")
print(f"   Probabilite de souscription : {proba*100:.1f}%")
print(f"   Decision : {'SOUSCRIT (yes)' if pred == 1 else 'NON SOUSCRIT (no)'}")

print("\nScores de tous les modeles :")
print("-" * 40)
for name, v in sorted(results.items(), key=lambda x: x[1]['test_auc'], reverse=True):
    clf_i = v['clf']
    X_i   = scaler.transform(nouveau_client) if v['use_sc'] else nouveau_client
    p     = clf_i.predict_proba(X_i)[0, 1]
    bar   = "#" * int(p * 20)
    print(f"  {name:<22} {p*100:5.1f}%  {bar}")


---
## 11. Conclusion

### Modifications apportees par rapport a la version initiale

| Modification | Justification |
|---|---|
| Suppression de `duree` | Data leakage : variable inconnue avant l'appel, inutilisable en production |
| Sauvegarde des `encoders` | Reutilisation coherente lors du scoring d'un nouveau client |
| Encodage cyclique `mois_sin` / `mois_cos` | Les mois sont cycliques : decembre est proche de janvier |
| SMOTE sur le train uniquement | Corrige le desequilibre 88%/12% sans contaminer le test |
| GridSearchCV sur Random Forest | Optimisation systematique des hyperparametres par validation croisee |
| Utilisation du meilleur RF pour Feature Importance | Importance calculee avec les hyperparametres optimaux |

### Reponses aux questions du projet

1. **Quels classifieurs sont utiles en banque ?**
   - Regression Logistique : interpretabilite reglementaire, coefficients lisibles
   - Random Forest / Gradient Boosting : haute performance sur donnees tabulaires
   - MLP : capture des relations non-lineaires complexes

2. **Comment creer un ensemble de modeles ?**
   - `VotingClassifier(voting='soft')` combinant RF + GB + AdaBoost + LR
   - Le vote soft moyenne les probabilites, plus robuste que le vote majoritaire

3. **Comment creer un classificateur base sur un reseau de neurones ?**
   - `MLPClassifier(hidden_layer_sizes=(128,64,32), activation='relu', solver='adam', alpha=0.001)`
   - Avec early stopping et learning rate adaptatif pour eviter le surapprentissage

4. **Comment estimer la precision d'un modele ?**
   - Validation croisee `StratifiedKFold(n_splits=5)` preservant les proportions de classes
   - Metriques : AUC-ROC (discriminante), F1-Score (equilibre precision/rappel), classification_report

### Perspectives
- Utiliser **XGBoost / LightGBM** pour de meilleures performances
- Appliquer **SHAP values** pour l'explicabilite individuelle des predictions
- Optimiser avec **Optuna** (plus efficace que GridSearchCV)
- Deployer le modele via **Flask / FastAPI**
- Mettre en place un pipeline **sklearn.Pipeline** pour encapsuler preprocessing + modele

---
*Projet ANI-IA 4 — Prediction du comportement client — ENSP Douala — 2024/2025*
